# 07. Testing Strategy — Separate unit, integration, and eval

Agent testing works best when different test types do different jobs. This chapter separates deterministic unit tests, provider integration checks, and behavioral evaluations.

**Learning goals**
- Decide what belongs in unit, integration, and eval layers.
- Use fake models for fast deterministic tests.
- Keep expensive or stateful checks behind explicit gates.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage

model = FakeListChatModel(responses=["test response"])
model.invoke([HumanMessage(content="ping")]).content

## 7.1 Testing pyramid

The testing pyramid still applies to agent systems, but the top layer often contains evaluations rather than only end-to-end tests. Put cheap deterministic checks at the base.


## 7.2 Unit test pure functions first

Pure functions are the easiest place to build confidence. Test routing, formatting, and scoring logic before introducing model variability.


In [ ]:
def normalize_question(text: str) -> str:
    return " ".join(text.strip().lower().split())

assert normalize_question("  Hello   Agent  ") == "hello agent"
print("unit test passed")

## 7.3 Integration gate

Integration tests prove that external providers and runtime wiring still work. Keep them clearly marked because they may require credentials, network access, or cost.


In [ ]:
def can_run_openai_integration() -> bool:
    return bool(os.getenv("OPENAI_API_KEY")) and os.getenv("RUN_LIVE_TESTS") == "1"

print("run live integration:", can_run_openai_integration())

## 7.4 Eval checks behavior paths, not only answer strings

Agent evaluations should inspect behavior, not just final text. Tool calls, routing choices, and recovery paths often matter more than exact wording.


In [ ]:
trajectory = [
    {"role": "user", "content": "weather in Seoul"},
    {"role": "assistant", "tool_calls": [{"name": "get_weather"}]},
    {"role": "tool", "name": "get_weather", "content": "clear"},
]

assert trajectory[1]["tool_calls"][0]["name"] == "get_weather"

## 7.5 Separate CI and manual verification

CI should stay fast and reliable. Manual or scheduled runs can cover heavier evaluations, live providers, and mutation-prone resources.


---

## Summary

| Item | Content |
|---|---|
| **Covered** | fake models, unit tests, integration gates, evals, and smoke-test boundaries |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`unit-testing.md`](../../docs/langchain/test/unit-testing.md)
- [`integration-testing.md`](../../docs/langchain/test/integration-testing.md)
- [`evals.md`](../../docs/langchain/test/evals.md)
